# Quickstart — Conformal VaR Audit on S&P 500

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danpele/Conformal_Oracle/blob/main/python/examples/notebooks/quickstart_sp500.ipynb)

A quick introduction to `conformal-oracle`: static and rolling conformal audits
on S&P 500 daily returns using GJR-GARCH and (optionally) Lag-Llama.

**Google Colab:** Run the install cell below first. It will restart the runtime
automatically — this is normal. After restart, skip the install cell and run
from the imports cell onward.

For the full 9-forecaster × 24-asset replication of Table 4 from Pele et al. (2026),
see [`reproduce_table4_full.ipynb`](reproduce_table4_full.ipynb).

In [ ]:
!pip install -q conformal-oracle[benchmarks] yfinance

# Colab: restart runtime so freshly installed packages are picked up
import sys
if "google.colab" in sys.modules:
    import IPython
    IPython.Application.instance().kernel.do_shutdown(True)

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
from conformal_oracle import audit
from conformal_oracle.contrib.benchmarks import GJRGARCHForecaster

np.random.seed(2026)

In [ ]:
spy = yf.download("^GSPC", start="2018-01-01", end="2026-03-13")
returns = np.log(spy["Close"]).diff().dropna().squeeze()
print(f"Loaded {len(returns)} daily log-returns")
returns.tail()

In [ ]:
result_garch = audit(
    returns,
    GJRGARCHForecaster(window=250),
    alpha=0.01,
    mode="static",
)
print(result_garch.summary())

In [ ]:
try:
    from conformal_oracle.contrib.tsfm.lag_llama import LagLlamaForecaster
    result_lagllama = audit(
        returns,
        LagLlamaForecaster(),
        alpha=0.01,
        mode="static",
        warmup=512,
    )
    print(result_lagllama.summary())
except ImportError:
    print("Lag-Llama not installed. Install with:")
    print("  pip install conformal-oracle[lag_llama]")
    print("  pip install git+https://github.com/time-series-foundation-models/lag-llama.git")
    result_lagllama = None

## Comparison with paper

| Forecaster | Paper qV | Notebook qV | Paper Regime | Notebook Regime |
|:-----------|:---------|:------------|:-------------|:----------------|
| GJR-GARCH  | −0.005   | *(see above)* | SP         | *(see above)*   |
| Lag-Llama  |  0.008   | *(see above)* | SP         | *(see above)*   |

Fill in the notebook values from the `summary()` output above.
GJR-GARCH should match within ±0.002; Lag-Llama varies with Monte Carlo sampling
but the regime classification (signal-preserving) should be invariant.

In [ ]:
result_garch_roll = audit(
    returns,
    GJRGARCHForecaster(window=250),
    alpha=0.01,
    mode="rolling",
    window=250,
)
print(result_garch_roll.summary())

## Notes

- Numerical differences from the paper reflect Monte Carlo variation in TSFM
  sampling (Lag-Llama uses 1000 samples per forecast by default).
- Different historical data windows may shift qV estimates slightly; the regime
  classification (signal-preserving vs replacement) is robust.
- Full paper replication requires running the panel inference via
  `conformal_oracle.panel.audit_panel` on all 24 assets.
- See the [methodology docs](../../docs/methodology.md) and the
  [API reference](../../docs/api.md) for details.